In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import Pipeline

C:\Users\moham\AppData\Roaming\Python\Python310\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
df = pd.read_csv('feature_selection_data_1.csv')

In [3]:
df.head()

,Unnamed: 0,index,brand,ultra_hd,full_hd,Price,Rating,inches,display_type,wifi,...,Company_flipkart,ppi,resolution_width,resolution_height,fast_charging,ram,Spec_Score,size_category,charging_category,brand_category
0,0,1,tcl,1,0,10.668955,4.65,55.0,qled,1,...,1.0,3.0,1.0,2.0,1.0,1.0,3.0,1.0,2.0,big
1,1,2,lg,1,0,10.645425,4.55,55.0,led,1,...,1.0,3.0,1.0,2.0,1.0,1.0,2.0,1.0,2.0,big
2,2,3,xiaomi,1,0,10.518673,4.35,55.0,qled,1,...,0.0,3.0,1.0,2.0,2.0,1.0,2.0,1.0,1.0,big
3,3,4,lg,1,0,10.341742,4.40,43.0,led,1,...,1.0,4.0,1.0,2.0,1.0,1.0,2.0,0.0,2.0,big
4,4,5,philips,1,0,11.225243,4.05,75.0,qled,1,...,0.0,1.0,1.0,2.0,2.0,1.0,4.0,1.0,2.0,big


In [4]:
df = df.drop(columns=['Unnamed: 0'])
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [5]:
df.head()

,index,brand,ultra_hd,full_hd,Price,Rating,inches,display_type,wifi,rom,value_score,smart_features,Company_amazon,Company_croma,Company_flipkart,ppi,resolution_width,resolution_height,fast_charging,ram,Spec_Score,size_category,charging_category,brand_category
0,1,tcl,1,0,10.668955,4.65,55.0,qled,1,16.0,0.006272,4.0,0.0,0.0,1.0,3.0,1.0,2.0,1.0,1.0,3.0,1.0,2.0,big
1,2,lg,1,0,10.645425,4.55,55.0,led,1,19.2,0.005958,3.0,0.0,0.0,1.0,3.0,1.0,2.0,1.0,1.0,2.0,1.0,2.0,big
2,3,xiaomi,1,0,10.518673,4.35,55.0,qled,1,32.0,0.006466,4.0,1.0,0.0,0.0,3.0,1.0,2.0,2.0,1.0,2.0,1.0,1.0,big
3,4,lg,1,0,10.341742,4.40,43.0,led,1,19.2,0.007381,3.0,0.0,0.0,1.0,4.0,1.0,2.0,1.0,1.0,2.0,0.0,2.0,big
4,5,philips,1,0,11.225243,4.05,75.0,qled,1,32.0,0.003618,4.0,1.0,0.0,0.0,1.0,1.0,2.0,2.0,1.0,4.0,1.0,2.0,big


In [6]:
df.duplicated().sum()

0

In [7]:
df.isnull().sum()

index                0
brand                0
ultra_hd             0
full_hd              0
Price                0
Rating               0
inches               0
display_type         0
wifi                 0
rom                  0
value_score          0
smart_features       0
Company_amazon       0
Company_croma        0
Company_flipkart     0
ppi                  0
resolution_width     0
resolution_height    0
fast_charging        0
ram                  0
Spec_Score           0
size_category        0
charging_category    0
brand_category       0
dtype: int64

In [8]:
df = df.drop(columns = ['index'])
df.head()

,brand,ultra_hd,full_hd,Price,Rating,inches,display_type,wifi,rom,value_score,smart_features,Company_amazon,Company_croma,Company_flipkart,ppi,resolution_width,resolution_height,fast_charging,ram,Spec_Score,size_category,charging_category,brand_category
0,tcl,1,0,10.668955,4.65,55.0,qled,1,16.0,0.006272,4.0,0.0,0.0,1.0,3.0,1.0,2.0,1.0,1.0,3.0,1.0,2.0,big
1,lg,1,0,10.645425,4.55,55.0,led,1,19.2,0.005958,3.0,0.0,0.0,1.0,3.0,1.0,2.0,1.0,1.0,2.0,1.0,2.0,big
2,xiaomi,1,0,10.518673,4.35,55.0,qled,1,32.0,0.006466,4.0,1.0,0.0,0.0,3.0,1.0,2.0,2.0,1.0,2.0,1.0,1.0,big
3,lg,1,0,10.341742,4.40,43.0,led,1,19.2,0.007381,3.0,0.0,0.0,1.0,4.0,1.0,2.0,1.0,1.0,2.0,0.0,2.0,big
4,philips,1,0,11.225243,4.05,75.0,qled,1,32.0,0.003618,4.0,1.0,0.0,0.0,1.0,1.0,2.0,2.0,1.0,4.0,1.0,2.0,big


In [9]:
X = df.drop(columns = ['Price'])
y = df['Price']

In [10]:
y = np.log1p(y)

# Ordinal Encoder

In [11]:
cat_cols = ['brand', 'display_type', 'brand_category']

In [12]:
preprocessing = ColumnTransformer(transformers = [
    ('scaling', StandardScaler(), ['inches', 'Spec_Score', 'ram', 'rom', 'smart_features']),
    ('ord encoding', OrdinalEncoder(handle_unknown = 'use_encoded_value', unknown_value = -1), cat_cols),
#     ('ohe encoding', OneHotEncoder(handle_unknown = 'ignore', sparse_output = False), ['brand'])
], remainder='passthrough')

In [13]:
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA
pipeline = Pipeline([
    ('preprocessing', preprocessing),
    ('pca', PCA()),
    ('model', LinearRegression())
])

In [14]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y, cv=kfold, scoring='r2')

In [15]:
scores.mean()

0.9718821332611391

In [16]:
scores.std()

0.0036331036844015315

In [17]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [18]:
pipeline.fit(X_train, y_train)

,steps,"[('preprocessing', ...), ('pca', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('scaling', ...), ('ord encoding', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [19]:
y_pred = pipeline.predict(X_test)

In [20]:
y_pred = np.expm1(y_pred)

In [21]:
from sklearn.metrics import mean_absolute_error
mean_absolute_error(np.expm1(y_test),y_pred)

0.11045387509469784

In [22]:
def scorer(model, model_name):
    output = []
    output.append(model_name)
    
    pipeline = Pipeline([
    ('preprocessing', preprocessing),
    ('model', model)
    ])
    
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    pipeline.fit(X_train, y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output

In [23]:
## lr, ridge, lasso, svr, dst, rf, gbr, abr, etr, xgb, mlp

from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor, ExtraTreesRegressor
from sklearn.neural_network import MLPRegressor
import xgboost as xgb

In [24]:
model_dict = {
    'lr_model' : LinearRegression(),
    'ridge_model' : Ridge(),
    'lasso_model' : Lasso(),
    'svr_model' : SVR(),
    'dt_model' : DecisionTreeRegressor(),
    'rf_model' : RandomForestRegressor(),
    'gbr_model' : GradientBoostingRegressor(),
    'ad_model' : AdaBoostRegressor(),
    'et_model' : ExtraTreesRegressor(),
    'mlp_model' : MLPRegressor(),
    'xg_model' : xgb.XGBRegressor()
}

In [25]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model, model_name))

In [26]:
model_output

[['lr_model', 0.9718821332611384, 0.11045387509469883],
 ['ridge_model', 0.8845875228415286, 0.22261513866848312],
 ['lasso_model', -0.005048794105982024, 0.7073754959019931],
 ['svr_model', 0.5172654040681128, 0.48733104915350195],
 ['dt_model', 0.9843704892796327, 0.06644203387452409],
 ['rf_model', 0.992878187327795, 0.04818849608845484],
 ['gbr_model', 0.9938188761921936, 0.046589705352980834],
 ['ad_model', 0.9780871474506064, 0.10382689427016199],
 ['et_model', 0.9933968020226468, 0.047309720700636035],
 ['mlp_model', -0.9719844796683187, 0.8789311886861352],
 ['xg_model', 0.9935765328130559, 0.04481933070011662]]

In [27]:
model_df = pd.DataFrame(model_output,
                       columns = ['name', 'r2', 'mae'])

In [28]:
model_df.sort_values(['mae'])

,name,r2,mae
10,xg_model,0.993577,0.044819
6,gbr_model,0.993819,0.046590
8,et_model,0.993397,0.047310
5,rf_model,0.992878,0.048188
4,dt_model,0.984370,0.066442
7,ad_model,0.978087,0.103827
0,lr_model,0.971882,0.110454
1,ridge_model,0.884588,0.222615
3,svr_model,0.517265,0.487331
2,lasso_model,-0.005049,0.707375


# OneHotEncoder

In [29]:
preprocessing1 = ColumnTransformer(transformers = [
    ('scaling', StandardScaler(), ['inches', 'Spec_Score', 'ram', 'rom', 'smart_features']),
    ('ord encoding', OrdinalEncoder(handle_unknown = 'use_encoded_value', unknown_value = -1), ['display_type', 'brand_category']),
    ('ohe encoding', OneHotEncoder(handle_unknown = 'ignore', sparse_output = False), ['brand'])
], remainder='passthrough')

In [30]:
from sklearn.decomposition import PCA
pipeline1 = Pipeline([
    ('preprocessing', preprocessing1),
    ('pca', PCA()),
    ('model', LinearRegression())
])

In [31]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline1, X, y, cv=kfold, scoring='r2')

In [32]:
scores.mean()

0.9773679445424529

In [33]:
scores.std()

0.0037566171824634943

In [34]:
pipeline1.fit(X_train, y_train)

,steps,"[('preprocessing', ...), ('pca', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('scaling', ...), ('ord encoding', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [35]:
y_pred = pipeline1.predict(X_test)

In [36]:
y_pred = np.expm1(y_pred)

In [37]:
mean_absolute_error(np.expm1(y_test), y_pred)

0.09901025479970331

In [38]:
def scorer(model, model_name):
    output = []
    output.append(model_name)
    
    pipeline1 = Pipeline([
    ('preprocessing', preprocessing1),
    ('model', model)
    ])
    
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline1, X, y, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    pipeline1.fit(X_train, y_train)
    
    y_pred = pipeline1.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output

In [39]:
model_dict = {
    'lr_model' : LinearRegression(),
    'ridge_model' : Ridge(),
    'lasso_model' : Lasso(),
    'svr_model' : SVR(),
    'dt_model' : DecisionTreeRegressor(),
    'rf_model' : RandomForestRegressor(),
    'gbr_model' : GradientBoostingRegressor(),
    'ad_model' : AdaBoostRegressor(),
    'et_model' : ExtraTreesRegressor(),
    'mlp_model' : MLPRegressor(),
    'xg_model' : xgb.XGBRegressor()
}

In [40]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model, model_name))

In [41]:
model_df = pd.DataFrame(model_output,
                       columns = ['name', 'r2', 'mae'])
model_df.sort_values(['mae'])

,name,r2,mae
10,xg_model,0.993124,0.044854
5,rf_model,0.992849,0.046932
6,gbr_model,0.993749,0.047443
8,et_model,0.992857,0.048058
4,dt_model,0.986037,0.066600
0,lr_model,0.977368,0.099010
7,ad_model,0.976981,0.110511
1,ridge_model,0.946128,0.151445
3,svr_model,0.486191,0.511557
2,lasso_model,-0.005049,0.707375


In [42]:
from sklearn.model_selection import GridSearchCV

In [43]:
param_grid = {
    'model__n_estimators': [50, 100, 150, 200],
    'model__max_depth': [None, 10, 20, 30],
    'model__max_samples': [0.1, 0.25, 0.50, 0.75],
    'model__max_features': ['sqrt', 'log2']
}

In [44]:
preprocessing2 = ColumnTransformer(transformers = [
    ('scaling', StandardScaler(), ['inches', 'Spec_Score', 'ram', 'rom', 'smart_features']),
    ('ord encoding', OrdinalEncoder(handle_unknown = 'use_encoded_value', unknown_value = -1), ['display_type', 'brand_category']),
    ('ohe encoding', OneHotEncoder(handle_unknown = 'ignore', sparse_output = False), ['brand'])
], remainder='passthrough')

In [45]:
pipeline2 = Pipeline([
    ('preprocessing', preprocessing2),
    ('model', RandomForestRegressor(random_state=42))
])

In [46]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
search = GridSearchCV(
    pipeline2,
    param_grid,
    cv=kfold,
    scoring='r2',
    n_jobs=-1
)

In [47]:
search.fit(X, y)

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'model__max_depth': [None, 10, ...], 'model__max_features': ['sqrt', 'log2'], 'model__max_samples': [0.1, 0.25, ...], 'model__n_estimators': [50, 100, ...]}"
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('scaling', ...), ('ord encoding', ...), ...]"


# Exporting Data

In [48]:
preprocessing2 = ColumnTransformer(transformers = [
    ('scaling', StandardScaler(), ['inches', 'Spec_Score', 'ram', 'rom', 'smart_features']),
    ('ord encoding', OrdinalEncoder(handle_unknown = 'use_encoded_value', unknown_value = -1), ['display_type', 'brand_category']),
    ('ohe encoding', OneHotEncoder(handle_unknown = 'ignore', sparse_output = False), ['brand'])
], remainder='passthrough')

In [49]:
pipeline2 = Pipeline([
    ('preprocessing', preprocessing2),
    ('model', RandomForestRegressor(random_state=42))
])

In [50]:
pipeline2.fit(X, y)

,steps,"[('preprocessing', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('scaling', ...), ('ord encoding', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [51]:
import pickle

In [52]:
with open('pipeline.pkl', 'wb') as file:
    pickle.dump(pipeline2, file)

In [53]:
with open('df.pkl', 'wb') as file:
    pickle.dump(X, file)

In [54]:
X

,brand,ultra_hd,full_hd,Rating,inches,display_type,wifi,rom,value_score,smart_features,Company_amazon,Company_croma,Company_flipkart,ppi,resolution_width,resolution_height,fast_charging,ram,Spec_Score,size_category,charging_category,brand_category
0,tcl,1,0,4.65,55.00,qled,1,16.0,0.006272,4.0,0.0,0.0,1.0,3.0,1.0,2.0,1.0,1.0,3.0,1.0,2.0,big
1,lg,1,0,4.55,55.00,led,1,19.2,0.005958,3.0,0.0,0.0,1.0,3.0,1.0,2.0,1.0,1.0,2.0,1.0,2.0,big
2,xiaomi,1,0,4.35,55.00,qled,1,32.0,0.006466,4.0,1.0,0.0,0.0,3.0,1.0,2.0,2.0,1.0,2.0,1.0,1.0,big
3,lg,1,0,4.40,43.00,led,1,19.2,0.007381,3.0,0.0,0.0,1.0,4.0,1.0,2.0,1.0,1.0,2.0,0.0,2.0,big
4,philips,1,0,4.05,75.00,qled,1,32.0,0.003618,4.0,1.0,0.0,0.0,1.0,1.0,2.0,2.0,1.0,4.0,1.0,2.0,big
5,hisense,1,0,4.20,55.00,miniled,1,28.8,0.005796,4.0,0.0,0.0,1.0,3.0,1.0,2.0,1.0,1.0,4.0,1.0,2.0,big
6,sony,1,0,4.50,65.00,led,1,16.0,0.003445,4.0,1.0,0.0,0.0,2.0,1.0,2.0,1.0,1.0,3.0,1.0,2.0,big
7,croma,0,0,4.45,32.00,led,1,16.0,0.014956,4.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,2.0,medium
8,realme,1,0,4.20,55.00,qled,1,16.0,0.010267,4.0,0.0,0.0,1.0,3.0,1.0,2.0,3.0,1.0,2.0,1.0,1.0,big
9,xiaomi,1,0,4.65,55.00,miniled,1,32.0,0.006303,4.0,1.0,0.0,0.0,3.0,1.0,2.0,1.0,1.0,3.0,1.0,2.0,big


# Trying to Prediction

In [55]:
X.columns

Index(['brand', 'ultra_hd', 'full_hd', 'Rating', 'inches', 'display_type',
       'wifi', 'rom', 'value_score', 'smart_features', 'Company_amazon',
       'Company_croma', 'Company_flipkart', 'ppi', 'resolution_width',
       'resolution_height', 'fast_charging', 'ram', 'Spec_Score',
       'size_category', 'charging_category', 'brand_category'],
      dtype='object')

In [56]:
X.iloc[0].values

array(['tcl', 1, 0, 4.65, 55.0, 'qled', 1, 16.0, 0.0062722388892764, 4.0,
       0.0, 0.0, 1.0, 3.0, 1.0, 2.0, 1.0, 1.0, 3.0, 1.0, 2.0, 'big'],
      dtype=object)

In [57]:
pipeline.named_steps['preprocessing'].feature_names_in_

array(['brand', 'ultra_hd', 'full_hd', 'Rating', 'inches', 'display_type',
       'wifi', 'rom', 'value_score', 'smart_features', 'Company_amazon',
       'Company_croma', 'Company_flipkart', 'ppi', 'resolution_width',
       'resolution_height', 'fast_charging', 'ram', 'Spec_Score',
       'size_category', 'charging_category', 'brand_category'],
      dtype=object)

In [58]:
data = [['lg', 0, 0, 4.0, 55, 'led', 1, 1.0, 0, 4, 0, 0, 1, 3, 2, 1, 2, 1, 4, 1, 2, 'big']]

columns = ['brand', 'ultra_hd', 'full_hd', 'Rating', 'inches', 'display_type',
       'wifi', 'rom', 'value_score', 'smart_features', 'Company_amazon',
       'Company_croma', 'Company_flipkart', 'ppi', 'resolution_width',
       'resolution_height', 'fast_charging', 'ram', 'Spec_Score',
       'size_category', 'charging_category', 'brand_category']

one_df = pd.DataFrame(data, columns = X.columns)
one_df

,brand,ultra_hd,full_hd,Rating,inches,display_type,wifi,rom,value_score,smart_features,Company_amazon,Company_croma,Company_flipkart,ppi,resolution_width,resolution_height,fast_charging,ram,Spec_Score,size_category,charging_category,brand_category
0,lg,0,0,4.0,55,led,1,1.0,0,4,0,0,1,3,2,1,2,1,4,1,2,big


In [59]:
np.expm1(pipeline.predict(one_df))[0]

11.660442031193714